# 📝 Práctica M05: Auditoría Estadística de una Cartera

**Escenario:** El departamento de Auditoría Interna te ha entregado un extracto de la base de datos de préstamos personales. Sospechan que hay anomalías en los datos y necesitan que valides la salud estadística de la cartera antes de entrenar cualquier modelo de Machine Learning.

**Ejercicio 0: Carga de datos**

Ejecuta esta celda para generar el dataset que vas a auditar. Contiene ingresos declarados por los clientes y el monto de préstamo solicitado.

In [9]:
import pandas as pd
import numpy as np

np.random.seed(123)

# Generamos datos realistas con algunas anomalías deliberadas
ingresos = np.concatenate([
    np.random.normal(2500, 500, 100),   # 100 clientes normales
    [25000, 32000, 45000],             # 3 Clientes con ingresos muy altos (Outliers)
    [10, 20, -500]                      # 3 Errores de carga de datos (Incoherencias)
])

prestamos = ingresos * 1.5 + np.random.normal(0, 1000, len(ingresos))

df_auditoria = pd.DataFrame({
    'Ingresos': ingresos,
    'Monto_Prestamo': prestamos
})

df_auditoria

,Ingresos,Monto_Prestamo
0,1957.184698,3577.831737
1,2998.672723,2520.121153
2,2641.489249,4674.498509
3,1746.852643,5218.582892
4,2210.699874,3291.423830
...,...,...
101,32000.000000,47401.894669
102,45000.000000,69700.702099
103,10.000000,703.296930
104,20.000000,23.692749


## 🛠️ EJERCICIOS

**Ejercicio 1: Detección de anomalías térmicas (Outliers)**

Calcula la Media y la Mediana de la columna Ingresos.
- ¿Cuál es mayor?
- Explica brevemente por qué existe esa diferencia y cuál de las dos medidas usarías para describir al "cliente típico".

In [7]:
df_auditoria.agg({
    'Ingresos': ['mean','median']
})
#Así podríamos devolver una tabla resumen.

media_ing = df_auditoria['Ingresos'].mean()
mediana_ing = df_auditoria['Ingresos'].median()
print(f"Media: {media_ing:.2f} | Mediana: {mediana_ing:.2f}")
#La media está sesgada, tiene muy en cuenta el valor de los extremos y atípicos
#lo que modifica su valor.
#Mientras tanto la mediana es más representativa y más robusta
#ante outliers, idóneo para representar el 'cliente típico'.


Media: 3329.11 | Mediana: 2473.37


**Ejercicio 2: Limpieza basada en lógica estadística**

Los ingresos negativos o cercanos a cero son errores de sistema.
1. Filtra el DataFrame para quedarte solo con clientes que tengan un ingreso mayor a 500€.
2. Calcula la Desviación Estándar antes y después de este filtro. ¿Cómo ha cambiado la dispersión de los datos?

In [ ]:
std_antes = df_auditoria['Ingresos'].std()
df_limpio = df_auditoria[df_auditoria['Ingresos'] > 500].copy()
std_despues = df_limpio['Ingresos'].std()

print(f"La diferencia entre la desviación antes y después es: \n")
print(f"{std_antes} | frente a | {std_despues}")
#Vemos que la desviación estándar no es estable frente a outliers
#pues a pesar de eliminar los que están por debajo de 500,
#estos son muy cercanos a la media y no la modificaban significativamente, 
#vemos que la std aumenta, esto es porque se ve muy sesgada por los 
#outliers extremos muy alejados de la media de nuestros datos,
#la cual se ve inflada por estos mismos.
#Al eliminar estos tres valores atípicos hemos reducido el nº de 
#la muestra pero no eliminado los valores que hacen una varianza extrema 


La diferencia entre la desviación antes y después es: 

5488.581653107776 | frente a | 5535.434031106472


**Ejercicio 3: Análisis de Concentración (Cuartiles)**

Calcula los cuartiles (Q1, Q2, Q3) de la columna Monto_Prestamo.
1. ¿Cuál es el monto máximo del 25% de los préstamos más pequeños?
2. ¿A partir de qué monto se encuentra el 25% de los préstamos más grandes (Top Tier)?

In [61]:
q1 = df_limpio['Monto_Prestamo'].quantile(0.25)
q2 = df_limpio['Monto_Prestamo'].quantile(0.50)
q3 = df_limpio['Monto_Prestamo'].quantile(0.75)

#print(f"El monto máximo del 25% de los préstamos más pequeños es: {q1:.2f} ")
#print(f"El monto a partir del cual se encuentra el 25% de los préstamos más grandes es: {q3:.2f}")

cuartiles = df_limpio['Monto_Prestamo'].quantile([0.25, 0.5, 0.75])
#print("\nDistribución de Préstamos:")
#print(cuartiles)
print(f"\nEl monto máximo del 25% de los préstamos más pequeños es: {cuartiles[0.25]:.2f}")
print(f"El monto a partir del cual se encuentra el 25% de los préstamos más grandes es: {cuartiles[0.75]:.2f}")


El monto máximo del 25% de los préstamos más pequeños es: 2947.90
El monto a partir del cual se encuentra el 25% de los préstamos más grandes es: 4722.03


**Ejercicio 4: Correlación Riesgo-Ingreso**

Calcula la Matriz de Correlación entre Ingresos y Monto_Prestamo.

- ¿Es una correlación positiva o negativa?
- ¿Qué significa esto para el negocio? (Ej: ¿A mayor ingreso, el banco presta más dinero?).

In [67]:
correlacion_ingresos_prestamo = df_limpio[['Ingresos', 'Monto_Prestamo']].corr()
print(f"La correlación entre Ingresos y Préstamo es de: {correlacion_ingresos_prestamo['Ingresos']['Monto_Prestamo']:.4f}")
#La correlación es positiva, por lo que si los ingresos aumentan aumentará el préstamo otorgado a ese cliente.

La correlación entre Ingresos y Préstamo es de: 0.9934


**Ejercicio 5: Visualización Conceptual (Sin código de gráficos aún)**

Si tuvieras que presentar un informe: ¿Qué valor de Asimetría (Skewness) esperarías encontrar en los ingresos de un banco: positivo, negativo o cero? Justifica tu respuesta.

In [59]:
#Se debería esperar una asimetría positiva (>0) es decir, 
#una cola larga hacia la derecha indicando personas con ingresos muy altos 
#y una masa grande de personas con salarios medios/bajos.

#La justificación radica en la naturaleza de la distribución de la riqueza:
#existe un 'suelo' o límite inferior rígido (los ingresos no pueden ser menores a cero), pero no existe un techo. 
#La gran mayoría de los clientes se agruparán en torno a salarios medios y bajos (ej. 1.000€ - 3.000€). 
#Sin embargo, un pequeño porcentaje de clientes tendrá ingresos extremadamente altos (decenas o cientos de miles de euros). 
#Estos valores extremos tiran de la 'cola' de la distribución hacia la derecha, 
#provocando que la media aritmética sea considerablemente mayor que la mediana.

## ✅ SOLUCIONES

In [ ]:
# 1. Media vs Mediana
media_ing = df_auditoria['Ingresos'].mean()
mediana_ing = df_auditoria['Ingresos'].median()
print(f"Media: {media_ing:.2f} | Mediana: {mediana_ing:.2f}")
# Respuesta: La media es mayor debido a los outliers. La mediana es más representativa.

# 2. Filtrado y Desviación
std_antes = df_auditoria['Ingresos'].std()
df_limpio = df_auditoria[df_auditoria['Ingresos'] > 500].copy()
std_despues = df_limpio['Ingresos'].std()
print(f"Desviación original: {std_antes:.2f} | Desviación tras limpieza: {std_despues:.2f}")

# 3. Cuartiles
cuartiles = df_limpio['Monto_Prestamo'].quantile([0.25, 0.5, 0.75])
print("\nDistribución de Préstamos:")
print(cuartiles)

# 4. Correlación
correlacion = df_limpio.corr()
print("\nMatriz de Correlación:")
print(correlacion)
# Respuesta: Positiva fuerte. El banco aumenta el préstamo según la capacidad de ingreso.

# 5. Respuesta Teórica Ejercicio 5
# Se espera Asimetría Positiva (> 0). Hay una masa grande de personas con salarios medios/bajos y una "cola" larga hacia la derecha de personas con ingresos muy altos.